# Pandas Part 2 Expanded: Missing Values, Replacement, and Data Cleaning

This expanded notebook builds on the Pandas Part 2 cleaning tutorial. It adds detailed explanations and examples for:

- `isna()`, `notna()`, `.any()`, `.all()`, `.sum()`, and missing-value percentages
- inspecting rows and columns with missing values
- replacing placeholder values with real `NaN`
- filling missing values with constants, mean, median, mode, group-based medians, and business formulas
- dropping missing values safely
- creating missing-value flags
- cleaning whitespace, capitalization, and inconsistent category labels
- complete exercises with solutions

Assumption: the dataset file is named `retail_data_messy.csv`. Change the file name below if needed.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_FILE = 'retail_data_messy.csv'

## 1. Load the dataset

Keep the raw data unchanged and work on a copy. This is a best practice because cleaning often requires trial and error.

In [ ]:
raw_df = pd.read_csv(DATA_FILE)
df = raw_df.copy()

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 2. Missing values: the basic tools

`df.isna()` returns `True` where a value is missing. `df.notna()` returns the opposite. These methods are the foundation of missing-value auditing.

In [ ]:
df.isna().head()

In [ ]:
df.notna().head()

### Count missing values by column

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts

### Check whether each column has any missing values

`.any()` gives a yes/no answer. This is useful when you do not need exact counts.

In [ ]:
df.isna().any()

In [ ]:
# Only show columns that have at least one missing value
df.columns[df.isna().any()]

In [ ]:
# Count how many columns have at least one missing value
df.isna().any().sum()

### Missing-value percentages

Counts alone can be misleading. Percentages tell us how serious the problem is relative to the dataset size.

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
}).sort_values('missing_count', ascending=False)

missing_summary

## 3. Missing values by row

Use `axis=1` when you want to evaluate each row across columns.

In [ ]:
# Rows with at least one missing value
rows_with_any_missing = df[df.isna().any(axis=1)]
rows_with_any_missing.head(10)

In [ ]:
print('Rows with at least one missing value:', rows_with_any_missing.shape[0])
print('Total rows:', df.shape[0])
print('Percent of rows with missing values:', rows_with_any_missing.shape[0] / df.shape[0] * 100)

In [ ]:
# Rows with no missing values
complete_rows = df[df.notna().all(axis=1)]
complete_rows.head()

In [ ]:
# Rows where every value is missing; usually rare but worth checking
df[df.isna().all(axis=1)]

## 4. Inspect important missing values

Do not immediately fill or drop missing values. First inspect the affected rows.

In [ ]:
# Inspect rows with missing revenue
if 'revenue' in df.columns:
    display(df[df['revenue'].isna()].head(10))

In [ ]:
# Missing revenue OR missing product_name
if {'revenue', 'product_name'}.issubset(df.columns):
    display(df[df['revenue'].isna() | df['product_name'].isna()].head(10))

In [ ]:
# Missing revenue AND missing product_name
if {'revenue', 'product_name'}.issubset(df.columns):
    display(df[df['revenue'].isna() & df['product_name'].isna()].head(10))

## 5. Placeholder values are not always recognized as missing

Real missing values are `NaN`. But messy datasets often contain placeholders such as empty strings, `N/A`, `unknown`, `null`, `?`, or `-`. These should usually be converted to `np.nan`.

In [ ]:
# Inspect frequent values in text columns
object_cols = df.select_dtypes(include='object').columns

for col in object_cols:
    print('
' + '='*70)
    print(col)
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
placeholder_missing_values = [
    '', ' ', '  ',
    'N/A', 'NA', 'n/a', 'na',
    'null', 'Null', 'NULL',
    'None', 'none',
    'unknown', 'Unknown', 'UNKNOWN',
    'missing', 'Missing', '?', '-'
]

df = df.replace(placeholder_missing_values, np.nan)

df.isna().sum().sort_values(ascending=False)

## 6. Replace specific dirty values

Use `.replace()` to standardize inconsistent labels. This is different from filling missing values.

In [ ]:
# Example: standardize channel labels
if 'channel' in df.columns:
    df['channel'] = df['channel'].replace({
        'online': 'Online',
        'ONLINE': 'Online',
        'Online ': 'Online',
        'store': 'Store',
        'STORE': 'Store',
        'Store ': 'Store'
    })
    df['channel'].value_counts(dropna=False)

In [ ]:
# Example: replace dirty labels in one column only
if 'segment' in df.columns:
    df['segment'] = df['segment'].replace({
        'Consumer ': 'Consumer',
        'consumer': 'Consumer',
        'CORPORATE': 'Corporate'
    })
    df['segment'].value_counts(dropna=False)

## 7. Clean whitespace and capitalization

Many category problems come from extra spaces and inconsistent capitalization. Use `.str.strip()` and `.str.title()` or `.str.lower()`.

In [ ]:
# Strip whitespace from all text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Standardize selected categorical columns to title case
for col in ['category', 'channel', 'segment', 'region', 'country', 'city']:
    if col in df.columns:
        df[col] = df[col].str.title()

for col in ['category', 'channel', 'segment', 'region']:
    if col in df.columns:
        print('
', col)
        print(df[col].value_counts(dropna=False).head(20))

## 8. Fill missing categorical values

For non-critical categorical columns, a common strategy is to fill missing values with `Unknown`. This avoids deleting rows while honestly preserving uncertainty.

In [ ]:
df_fill_cat = df.copy()

for col in ['segment', 'city', 'channel', 'product_name']:
    if col in df_fill_cat.columns:
        df_fill_cat[col] = df_fill_cat[col].fillna('Unknown')

df_fill_cat[['segment', 'city', 'channel', 'product_name']].isna().sum()

## 9. Fill numeric missing values

Numeric columns need more care. Common choices include:

- mean: useful when distribution is roughly symmetric
- median: safer when there are outliers
- group median: better when values differ by group
- formula-based fill: best when the value can be calculated from other columns

In [ ]:
# Fill shipping_cost with median
if 'shipping_cost' in df.columns:
    df_num = df.copy()
    median_shipping = df_num['shipping_cost'].median()
    df_num['shipping_cost_median_filled'] = df_num['shipping_cost'].fillna(median_shipping)
    print('Median shipping cost:', median_shipping)
    display(df_num[['shipping_cost', 'shipping_cost_median_filled']].head(10))

In [ ]:
# Fill discount with mean, if needed
if 'discount' in df.columns:
    df_num = df.copy()
    mean_discount = df_num['discount'].mean()
    df_num['discount_mean_filled'] = df_num['discount'].fillna(mean_discount)
    print('Mean discount:', mean_discount)
    display(df_num[['discount', 'discount_mean_filled']].head(10))

## 10. Fill using mode

Mode is the most common value. It is often used for categorical variables.

In [ ]:
if 'channel' in df.columns:
    channel_mode = df['channel'].mode(dropna=True)[0]
    df_mode = df.copy()
    df_mode['channel_mode_filled'] = df_mode['channel'].fillna(channel_mode)
    print('Most common channel:', channel_mode)
    display(df_mode[['channel', 'channel_mode_filled']].head(10))

## 11. Group-based imputation

If shipping cost differs by region, filling with the regional median is more meaningful than using one overall median.

In [ ]:
df_group = df.copy()

if {'shipping_cost', 'region'}.issubset(df_group.columns):
    df_group['shipping_cost_region_filled'] = df_group['shipping_cost'].fillna(
        df_group.groupby('region')['shipping_cost'].transform('median')
    )
    df_group['shipping_cost_region_filled'] = df_group['shipping_cost_region_filled'].fillna(
        df_group['shipping_cost'].median()
    )
    display(df_group[['region', 'shipping_cost', 'shipping_cost_region_filled']].head(10))
    print(df_group[['shipping_cost', 'shipping_cost_region_filled']].isna().sum())

## 12. Fill revenue using a business formula

If revenue should equal `quantity * unit_price * (1 - discount)`, use the formula instead of mean or median. This is usually a better analytical decision.

In [ ]:
df_formula = df.copy()
required_cols = {'revenue', 'quantity', 'unit_price', 'discount'}

if required_cols.issubset(df_formula.columns):
    calculated_revenue = df_formula['quantity'] * df_formula['unit_price'] * (1 - df_formula['discount'])
    df_formula['revenue_filled'] = df_formula['revenue'].fillna(calculated_revenue)
    display(df_formula[df_formula['revenue'].isna()][['quantity', 'unit_price', 'discount', 'revenue', 'revenue_filled']].head(10))
    print('Missing original revenue:', df_formula['revenue'].isna().sum())
    print('Missing filled revenue:', df_formula['revenue_filled'].isna().sum())

## 13. Missing-value flags

Sometimes the fact that a value is missing is meaningful. Before filling values, create missing-value flags.

In [ ]:
df_flags = df.copy()

for col in ['segment', 'shipping_cost', 'city', 'channel', 'product_name', 'revenue']:
    if col in df_flags.columns:
        df_flags[col + '_missing'] = df_flags[col].isna()

flag_cols = [col for col in df_flags.columns if col.endswith('_missing')]
display(df_flags[flag_cols].head())
display(df_flags[flag_cols].sum().sort_values(ascending=False))

## 14. Drop missing values carefully

Dropping rows is simple, but it can throw away useful data. Use it when the missing value makes the row unusable for your analysis.

In [ ]:
# Drop rows where all columns are missing
no_empty_rows = df.dropna(how='all')
print('Original rows:', df.shape[0])
print('Rows after dropping fully empty rows:', no_empty_rows.shape[0])

In [ ]:
# Drop rows where revenue is missing
if 'revenue' in df.columns:
    no_missing_revenue = df.dropna(subset=['revenue'])
    print('Rows after dropping missing revenue:', no_missing_revenue.shape[0])

In [ ]:
# Drop rows where either revenue or product_name is missing
if {'revenue', 'product_name'}.issubset(df.columns):
    no_missing_core = df.dropna(subset=['revenue', 'product_name'])
    print('Rows after dropping missing revenue or product_name:', no_missing_core.shape[0])

## 15. Convert data types after cleaning

Use `pd.to_numeric(..., errors='coerce')` for numeric columns and `pd.to_datetime(..., errors='coerce')` for date columns. Invalid values become missing values, which can then be handled.

In [ ]:
df_types = df.copy()

for col in ['quantity', 'discount', 'unit_price', 'shipping_cost', 'revenue']:
    if col in df_types.columns:
        df_types[col] = pd.to_numeric(df_types[col], errors='coerce')

if 'order_date' in df_types.columns:
    df_types['order_date'] = pd.to_datetime(df_types['order_date'], errors='coerce')

df_types.dtypes

## 16. A reusable cleaning pipeline

In [ ]:
def clean_retail_data(raw):
    df = raw.copy()

    placeholder_missing_values = [
        '', ' ', '  ', 'N/A', 'NA', 'n/a', 'na',
        'null', 'Null', 'NULL', 'None', 'none',
        'unknown', 'Unknown', 'UNKNOWN', 'missing', 'Missing', '?', '-'
    ]
    df = df.replace(placeholder_missing_values, np.nan)

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

    for col in ['category', 'channel', 'segment', 'region', 'country', 'city']:
        if col in df.columns:
            df[col] = df[col].str.title()

    for col in ['quantity', 'discount', 'unit_price', 'shipping_cost', 'revenue']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'order_date' in df.columns:
        df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

    for col in ['segment', 'shipping_cost', 'city', 'channel', 'product_name', 'revenue']:
        if col in df.columns:
            df[col + '_missing'] = df[col].isna()

    for col in ['segment', 'city', 'channel', 'product_name']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    if {'shipping_cost', 'region'}.issubset(df.columns):
        df['shipping_cost'] = df['shipping_cost'].fillna(
            df.groupby('region')['shipping_cost'].transform('median')
        )
        df['shipping_cost'] = df['shipping_cost'].fillna(df['shipping_cost'].median())

    if {'revenue', 'quantity', 'unit_price', 'discount'}.issubset(df.columns):
        revenue_formula = df['quantity'] * df['unit_price'] * (1 - df['discount'])
        df['revenue'] = df['revenue'].fillna(revenue_formula)

    return df

clean_df = clean_retail_data(raw_df)
clean_df.head()

In [ ]:
comparison = pd.DataFrame({
    'before_cleaning': raw_df.isna().sum(),
    'after_cleaning': clean_df.isna().sum()
}).sort_values('before_cleaning', ascending=False)

comparison

# Exercises with Solutions

## Exercise 1: Basic missing-value audit

**Task:** Count missing values, show columns with missing values, calculate percentages, and show the top 5 columns.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_columns = missing_counts[missing_counts > 0]
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

display(missing_columns)
display(missing_percent[missing_percent > 0])
display(missing_counts.head(5))

## Exercise 2: Use any and all

**Task:** Find rows with any missing value, rows with no missing values, columns with missing values, and columns with no missing values.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
rows_any_missing = df[df.isna().any(axis=1)]
rows_no_missing = df[df.notna().all(axis=1)]
cols_any_missing = df.columns[df.isna().any(axis=0)]
cols_no_missing = df.columns[df.notna().all(axis=0)]

print('Rows with any missing:', rows_any_missing.shape[0])
print('Rows with no missing:', rows_no_missing.shape[0])
print('Columns with any missing:', list(cols_any_missing))
print('Columns with no missing:', list(cols_no_missing))

## Exercise 3: Replace placeholders

**Task:** Replace placeholder values such as N/A, unknown, ?, and - with np.nan.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex3 = df.copy()
placeholders = ['', ' ', 'N/A', 'NA', 'unknown', 'Unknown', '?', '-', 'null', 'None']
df_ex3 = df_ex3.replace(placeholders, np.nan)
df_ex3.isna().sum().sort_values(ascending=False)

## Exercise 4: Fill categorical values

**Task:** Fill missing segment, city, and channel values with Unknown.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex4 = df.copy()
for col in ['segment', 'city', 'channel']:
    if col in df_ex4.columns:
        df_ex4[col] = df_ex4[col].fillna('Unknown')

df_ex4[[col for col in ['segment', 'city', 'channel'] if col in df_ex4.columns]].isna().sum()

## Exercise 5: Fill numeric values

**Task:** Fill missing shipping_cost using the median. Create a new column instead of overwriting the original.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex5 = df.copy()
if 'shipping_cost' in df_ex5.columns:
    median_shipping = df_ex5['shipping_cost'].median()
    df_ex5['shipping_cost_filled'] = df_ex5['shipping_cost'].fillna(median_shipping)
    display(df_ex5[['shipping_cost', 'shipping_cost_filled']].head(10))

## Exercise 6: Add missing-value flags

**Task:** Create flags for segment, shipping_cost, and revenue missingness.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex6 = df.copy()
for col in ['segment', 'shipping_cost', 'revenue']:
    if col in df_ex6.columns:
        df_ex6[col + '_missing'] = df_ex6[col].isna()

flag_cols = [col for col in df_ex6.columns if col.endswith('_missing')]
display(df_ex6[flag_cols].head())
display(df_ex6[flag_cols].sum())

## Exercise 7: Fill revenue using a formula

**Task:** Fill missing revenue using quantity * unit_price * (1 - discount).

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex7 = df.copy()
if {'revenue', 'quantity', 'unit_price', 'discount'}.issubset(df_ex7.columns):
    calculated_revenue = df_ex7['quantity'] * df_ex7['unit_price'] * (1 - df_ex7['discount'])
    df_ex7['revenue_filled'] = df_ex7['revenue'].fillna(calculated_revenue)
    display(df_ex7[df_ex7['revenue'].isna()][['quantity', 'unit_price', 'discount', 'revenue', 'revenue_filled']].head())

## Exercise 8: Group-based imputation

**Task:** Fill missing shipping_cost using region median, then overall median for remaining missing values.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex8 = df.copy()
if {'shipping_cost', 'region'}.issubset(df_ex8.columns):
    df_ex8['shipping_cost_region_filled'] = df_ex8['shipping_cost'].fillna(
        df_ex8.groupby('region')['shipping_cost'].transform('median')
    )
    df_ex8['shipping_cost_region_filled'] = df_ex8['shipping_cost_region_filled'].fillna(df_ex8['shipping_cost'].median())
    display(df_ex8[['region', 'shipping_cost', 'shipping_cost_region_filled']].head(10))

## Exercise 9: Clean text columns

**Task:** Strip whitespace and standardize category, channel, and segment using title case.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
df_ex9 = df.copy()
for col in df_ex9.select_dtypes(include='object').columns:
    df_ex9[col] = df_ex9[col].str.strip()

for col in ['category', 'channel', 'segment']:
    if col in df_ex9.columns:
        df_ex9[col] = df_ex9[col].str.title()
        print('
', col)
        print(df_ex9[col].value_counts(dropna=False).head(20))

## Exercise 10: Written interpretation

**Task:** Write a short paragraph explaining which missing values are tolerable, critical, and good candidates for flags.

Write your code in the blank cell below.

In [ ]:
# Your code here

### Solution

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
}).sort_values('missing_count', ascending=False)

missing_summary.head(10)

## Model written answer for Exercise 10

A strong answer might say:

The columns with the most missing values are the columns at the top of the missing-value summary, such as `segment`, `shipping_cost`, `city`, `channel`, `product_name`, and `revenue` if those columns appear in the dataset. Missing values in `segment`, `city`, and `channel` may be tolerable because these are descriptive fields that can often be grouped as `Unknown`. Missing `revenue` is more critical because revenue is a key outcome variable used in KPIs, aggregation, and modeling. Missing `product_name` is also important for product-level analysis. Columns such as `shipping_cost` and `segment` are good candidates for missing-value flags because the missingness itself may contain useful information.

# Part B: Creating New Columns for Feature Engineering

Data cleaning prepares the dataset. Feature engineering creates new variables that make later analysis or machine learning more useful.

A **new column** is usually created from:

- one existing column
- multiple existing columns
- conditional logic
- text processing
- dates
- group-level summaries
- custom functions

Important rule: avoid changing the raw dataset directly. Work on a copy.


In [ ]:
# Start from the cleaned DataFrame if it exists.
# If not, fall back to df.
try:
    feature_df = clean_df.copy()
except NameError:
    feature_df = df.copy()

feature_df.head()


## 17. Add a simple new column

The most common pattern is:

```python
DataFrame['new_column'] = calculation
```

For example, if `revenue`, `quantity`, and `unit_price` are available, we can create useful transaction-level features.


In [ ]:
# Add a revenue-per-unit column
# This protects against division by zero.
feature_df['revenue_per_unit'] = np.where(
    feature_df['quantity'] != 0,
    feature_df['revenue'] / feature_df['quantity'],
    np.nan
)

feature_df[['revenue', 'quantity', 'revenue_per_unit']].head(10)


### Why this matters

`revenue_per_unit` can help identify:

- unusually high-priced transactions
- possible data entry errors
- discounts or promotions
- product-level pricing differences


## 18. Add a column based on several columns

Many business features combine multiple fields. A common example is estimated gross revenue before discount.


In [ ]:
# Estimated gross sales before discount
feature_df['gross_sales'] = feature_df['unit_price'] * feature_df['quantity']

# Estimated discount amount
feature_df['discount_amount'] = feature_df['gross_sales'] * feature_df['discount']

# Estimated net sales after discount
feature_df['estimated_net_sales'] = feature_df['gross_sales'] - feature_df['discount_amount']

feature_df[['unit_price', 'quantity', 'discount', 'gross_sales', 'discount_amount', 'estimated_net_sales', 'revenue']].head(10)


### Teaching point

A new feature is not automatically correct. After creating it, compare it against existing columns.

Here, `estimated_net_sales` can be compared with `revenue`. If they differ, that may reveal:

- taxes
- shipping fees
- returns
- missing values
- data quality problems
- different business definitions of revenue


In [ ]:
feature_df['revenue_gap'] = feature_df['revenue'] - feature_df['estimated_net_sales']

feature_df[['revenue', 'estimated_net_sales', 'revenue_gap']].head(10)


## 19. Add binary indicator columns

Indicator columns are very useful for analysis and machine learning. They usually contain `True/False` or `0/1`.


In [ ]:
# Boolean indicators
feature_df['has_discount'] = feature_df['discount'] > 0
feature_df['is_zero_quantity'] = feature_df['quantity'] == 0
feature_df['is_negative_shipping'] = feature_df['shipping_cost'] < 0
feature_df['revenue_missing'] = feature_df['revenue'].isna()
feature_df['shipping_cost_missing'] = feature_df['shipping_cost'].isna()

feature_df[['discount', 'has_discount', 'quantity', 'is_zero_quantity', 'shipping_cost', 'is_negative_shipping', 'revenue', 'revenue_missing']].head(10)


### Convert Boolean flags to 0/1 if needed

Some machine learning tools prefer numeric columns. You can convert `True/False` to `1/0` using `.astype(int)`.


In [ ]:
feature_df['has_discount_int'] = feature_df['has_discount'].astype(int)
feature_df['shipping_cost_missing_int'] = feature_df['shipping_cost_missing'].astype(int)

feature_df[['has_discount', 'has_discount_int', 'shipping_cost_missing', 'shipping_cost_missing_int']].head()


## 20. Add a column using `np.where()`

`np.where()` is useful for simple if/else logic.

Pattern:

```python
np.where(condition, value_if_true, value_if_false)
```


In [ ]:
feature_df['discount_label'] = np.where(
    feature_df['discount'] > 0,
    'Discounted',
    'No Discount'
)

feature_df[['discount', 'discount_label']].head(10)


## 21. Add a column using multiple conditions with `np.select()`

Use `np.select()` when there are more than two categories.


In [ ]:
conditions = [
    feature_df['discount'] == 0,
    feature_df['discount'].between(0.01, 0.10, inclusive='both'),
    feature_df['discount'].between(0.11, 0.25, inclusive='both'),
    feature_df['discount'] > 0.25
]

choices = [
    'No discount',
    'Small discount',
    'Medium discount',
    'Large discount'
]

feature_df['discount_level'] = np.select(conditions, choices, default='Unknown')

feature_df[['discount', 'discount_level']].head(15)


### Check the result

Always check the distribution after creating categorical features.


In [ ]:
feature_df['discount_level'].value_counts(dropna=False)


## 22. Add columns from date variables

Date columns are powerful for feature engineering. First convert the column to datetime.


In [ ]:
feature_df['order_date'] = pd.to_datetime(feature_df['order_date'], errors='coerce')

feature_df['order_year'] = feature_df['order_date'].dt.year
feature_df['order_month'] = feature_df['order_date'].dt.month
feature_df['order_day'] = feature_df['order_date'].dt.day
feature_df['order_day_name'] = feature_df['order_date'].dt.day_name()
feature_df['order_quarter'] = feature_df['order_date'].dt.quarter
feature_df['is_weekend'] = feature_df['order_date'].dt.dayofweek >= 5

feature_df[['order_date', 'order_year', 'order_month', 'order_day_name', 'order_quarter', 'is_weekend']].head(10)


### Why date features matter

Date features allow questions such as:

- Which month has the highest revenue?
- Are weekend orders different from weekday orders?
- Are discounts more common in some quarters?
- Does shipping cost vary by month?


In [ ]:
feature_df.groupby('order_month')['revenue'].sum().sort_index()


## 23. Add columns from text variables

Text columns often contain useful information, but they are messy. Start with basic cleaning.


In [ ]:
# Standardize product names before using them
feature_df['product_name_clean'] = (
    feature_df['product_name']
    .astype('string')
    .str.strip()
    .str.lower()
)

# Add text-based features
feature_df['product_name_length'] = feature_df['product_name_clean'].str.len()
feature_df['product_name_missing'] = feature_df['product_name'].isna()

feature_df[['product_name', 'product_name_clean', 'product_name_length', 'product_name_missing']].head(10)


## 24. Add columns with `.assign()`

`.assign()` is another clean way to create new columns. It is useful when you want to chain operations.


In [ ]:
feature_df = feature_df.assign(
    profit_proxy = feature_df['revenue'] - feature_df['shipping_cost'],
    high_revenue_order = feature_df['revenue'] > feature_df['revenue'].median()
)

feature_df[['revenue', 'shipping_cost', 'profit_proxy', 'high_revenue_order']].head(10)


## 25. Add columns using a custom function

Sometimes the logic is too complex for one line. In that case, define a function and use `.apply()`.

This is easier to read, but it can be slower than vectorized pandas operations on very large datasets.


In [ ]:
def classify_order(row):
    """
    Classify an order based on revenue, discount, and quantity.
    This is a simple teaching example, not a universal business rule.
    """
    if pd.isna(row['revenue']):
        return 'Missing revenue'
    elif row['quantity'] <= 0:
        return 'Check quantity'
    elif row['revenue'] >= 500 and row['discount'] > 0:
        return 'High revenue with discount'
    elif row['revenue'] >= 500:
        return 'High revenue'
    elif row['discount'] > 0:
        return 'Discounted order'
    else:
        return 'Regular order'

feature_df['order_type'] = feature_df.apply(classify_order, axis=1)

feature_df[['revenue', 'quantity', 'discount', 'order_type']].head(15)


In [ ]:
feature_df['order_type'].value_counts(dropna=False)


## 26. Add group-based features with `groupby()` and `transform()`

Group-based features compare each row with its group. This is very useful for feature engineering.

Example: compare each order's revenue to the average revenue in the same category.


In [ ]:
feature_df['category_avg_revenue'] = feature_df.groupby('category')['revenue'].transform('mean')
feature_df['revenue_vs_category_avg'] = feature_df['revenue'] - feature_df['category_avg_revenue']

feature_df[['category', 'revenue', 'category_avg_revenue', 'revenue_vs_category_avg']].head(15)


### More group-based feature examples

These features are useful for identifying outliers and customer behavior patterns.


In [ ]:
feature_df['customer_order_count'] = feature_df.groupby('customer_id')['order_id'].transform('count')
feature_df['customer_total_revenue'] = feature_df.groupby('customer_id')['revenue'].transform('sum')
feature_df['region_avg_shipping'] = feature_df.groupby('region')['shipping_cost'].transform('mean')

feature_df[['customer_id', 'order_id', 'revenue', 'customer_order_count', 'customer_total_revenue', 'region', 'shipping_cost', 'region_avg_shipping']].head(15)


## 27. Add rank and percentile features

Ranking helps identify high-value orders, customers, or products.


In [ ]:
feature_df['revenue_rank'] = feature_df['revenue'].rank(ascending=False, method='dense')
feature_df['revenue_percentile'] = feature_df['revenue'].rank(pct=True)

feature_df[['order_id', 'revenue', 'revenue_rank', 'revenue_percentile']].sort_values('revenue_rank').head(10)


## 28. Add binned features with `pd.cut()` and `pd.qcut()`

Binning converts a numeric column into categories.

- `pd.cut()` uses manually defined ranges.
- `pd.qcut()` creates bins with roughly equal numbers of rows.


In [ ]:
feature_df['revenue_bin'] = pd.cut(
    feature_df['revenue'],
    bins=[-np.inf, 100, 300, 600, np.inf],
    labels=['Low', 'Medium', 'High', 'Very High']
)

feature_df[['revenue', 'revenue_bin']].head(10)


In [ ]:
# qcut can fail if there are too many duplicate values.
# duplicates='drop' makes it more robust.
feature_df['revenue_quartile'] = pd.qcut(
    feature_df['revenue'],
    q=4,
    labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'],
    duplicates='drop'
)

feature_df[['revenue', 'revenue_quartile']].head(10)


## 29. Add interaction features

Interaction features combine variables to capture relationships.

Examples:

- discount × revenue
- quantity × unit price
- weekend × channel
- segment × category


In [ ]:
feature_df['discount_x_revenue'] = feature_df['discount'] * feature_df['revenue']
feature_df['quantity_x_unit_price'] = feature_df['quantity'] * feature_df['unit_price']

feature_df[['discount', 'revenue', 'discount_x_revenue', 'quantity', 'unit_price', 'quantity_x_unit_price']].head(10)


## 30. Feature engineering checklist

Before using a new column, ask:

1. Does the feature make business sense?
2. Does it leak future information?
3. Does it duplicate another variable?
4. Does it create errors when values are missing?
5. Does it behave correctly when quantity is zero or negative?
6. Should missingness be turned into a feature?
7. Should the feature be numeric, categorical, or Boolean?
8. Is it useful for analysis, visualization, or modeling?


# Practice Tasks: New Columns and Feature Engineering

Use `feature_df` for the following tasks. Each task is followed by a solution section.


## Task 1

Create a new column called `net_revenue_after_shipping`:

```python
revenue - shipping_cost
```

Then display `revenue`, `shipping_cost`, and the new column.


In [ ]:
# Solution 1
feature_df['net_revenue_after_shipping'] = feature_df['revenue'] - feature_df['shipping_cost']

feature_df[['revenue', 'shipping_cost', 'net_revenue_after_shipping']].head(10)


## Task 2

Create a column called `large_order_flag` that is `True` when `quantity >= 10` and `False` otherwise.


In [ ]:
# Solution 2
feature_df['large_order_flag'] = feature_df['quantity'] >= 10

feature_df[['quantity', 'large_order_flag']].head(10)


## Task 3

Create a column called `shipping_status`:

- `Missing shipping` if `shipping_cost` is missing
- `Negative shipping` if `shipping_cost < 0`
- `Free shipping` if `shipping_cost == 0`
- `Paid shipping` otherwise


In [ ]:
# Solution 3
conditions = [
    feature_df['shipping_cost'].isna(),
    feature_df['shipping_cost'] < 0,
    feature_df['shipping_cost'] == 0
]

choices = [
    'Missing shipping',
    'Negative shipping',
    'Free shipping'
]

feature_df['shipping_status'] = np.select(conditions, choices, default='Paid shipping')

feature_df[['shipping_cost', 'shipping_status']].head(15)


## Task 4

Create date features from `order_date`:

- `month_name`
- `weekday_number`
- `is_month_end`


In [ ]:
# Solution 4
feature_df['order_date'] = pd.to_datetime(feature_df['order_date'], errors='coerce')

feature_df['month_name'] = feature_df['order_date'].dt.month_name()
feature_df['weekday_number'] = feature_df['order_date'].dt.dayofweek
feature_df['is_month_end'] = feature_df['order_date'].dt.is_month_end

feature_df[['order_date', 'month_name', 'weekday_number', 'is_month_end']].head(10)


## Task 5

Create a custom function called `revenue_quality()` that returns:

- `Missing` if revenue is missing
- `Negative` if revenue is less than 0
- `Zero` if revenue equals 0
- `Positive` otherwise

Apply it to create a new column called `revenue_quality`.


In [ ]:
# Solution 5
def revenue_quality(x):
    if pd.isna(x):
        return 'Missing'
    elif x < 0:
        return 'Negative'
    elif x == 0:
        return 'Zero'
    else:
        return 'Positive'

feature_df['revenue_quality'] = feature_df['revenue'].apply(revenue_quality)

feature_df[['revenue', 'revenue_quality']].head(15)


## Task 6

Create a group-based feature called `category_median_revenue`, then create `above_category_median` to show whether each order's revenue is above its category median.


In [ ]:
# Solution 6
feature_df['category_median_revenue'] = feature_df.groupby('category')['revenue'].transform('median')
feature_df['above_category_median'] = feature_df['revenue'] > feature_df['category_median_revenue']

feature_df[['category', 'revenue', 'category_median_revenue', 'above_category_median']].head(15)


## Task 7

Create a feature called `customer_avg_revenue` and then create `order_vs_customer_avg`.

This compares each order with that customer's average order revenue.


In [ ]:
# Solution 7
feature_df['customer_avg_revenue'] = feature_df.groupby('customer_id')['revenue'].transform('mean')
feature_df['order_vs_customer_avg'] = feature_df['revenue'] - feature_df['customer_avg_revenue']

feature_df[['customer_id', 'revenue', 'customer_avg_revenue', 'order_vs_customer_avg']].head(15)


## Task 8

Create a clean customer segment column called `segment_clean`:

- strip whitespace
- convert to lowercase
- replace missing values with `unknown`

Then show the frequency count.


In [ ]:
# Solution 8
feature_df['segment_clean'] = (
    feature_df['segment']
    .astype('string')
    .str.strip()
    .str.lower()
    .fillna('unknown')
)

feature_df['segment_clean'].value_counts(dropna=False)


## Task 9

Create a column called `potential_data_issue` that is `True` if any of the following are true:

- revenue is missing
- quantity is less than or equal to 0
- shipping cost is negative
- product name is missing


In [ ]:
# Solution 9
feature_df['potential_data_issue'] = (
    feature_df['revenue'].isna()
    | (feature_df['quantity'] <= 0)
    | (feature_df['shipping_cost'] < 0)
    | (feature_df['product_name'].isna())
)

feature_df[['revenue', 'quantity', 'shipping_cost', 'product_name', 'potential_data_issue']].head(20)


## Task 10

Create a compact feature-engineering summary table showing:

- number of rows
- number of columns
- number of columns added compared with the original `df`
- number of rows with potential data issues


In [ ]:
# Solution 10
summary = pd.DataFrame({
    'metric': [
        'rows',
        'columns in original df',
        'columns in feature_df',
        'new columns added',
        'rows with potential data issues'
    ],
    'value': [
        len(feature_df),
        df.shape[1],
        feature_df.shape[1],
        feature_df.shape[1] - df.shape[1],
        feature_df['potential_data_issue'].sum()
    ]
})

summary


# Assignment: Feature Engineering Mini Project

Use the retail dataset to create a feature-engineered dataset that could support later modeling or dashboard analysis.

## Requirements

Create at least **10 new columns**:

1. At least 2 missing-value flags
2. At least 2 date-based features
3. At least 2 numeric calculation features
4. At least 1 text-cleaning feature
5. At least 1 group-based feature using `groupby().transform()`
6. At least 1 custom function feature
7. At least 1 potential data-quality flag

## Deliverables

Submit a notebook section that includes:

- your code
- a short explanation of each new column
- a summary table of the new columns
- 3 business insights from the new features


## Assignment starter code

Students can start here and fill in the missing parts.


In [ ]:
assignment_df = df.copy()

# 1. Missing-value flags
assignment_df['revenue_missing'] = assignment_df['revenue'].isna()
assignment_df['shipping_missing'] = assignment_df['shipping_cost'].isna()

# 2. Date-based features
assignment_df['order_date'] = pd.to_datetime(assignment_df['order_date'], errors='coerce')
assignment_df['order_month'] = assignment_df['order_date'].dt.month
assignment_df['order_weekday'] = assignment_df['order_date'].dt.day_name()

# 3. Numeric calculation features
assignment_df['gross_sales'] = assignment_df['unit_price'] * assignment_df['quantity']
assignment_df['discount_amount'] = assignment_df['gross_sales'] * assignment_df['discount']

# 4. Text-cleaning feature
assignment_df['category_clean'] = assignment_df['category'].astype('string').str.strip().str.lower()

# 5. Group-based feature
assignment_df['category_avg_revenue'] = assignment_df.groupby('category_clean')['revenue'].transform('mean')

# 6. Custom function feature
def classify_discount(d):
    if pd.isna(d):
        return 'Unknown'
    elif d == 0:
        return 'No discount'
    elif d <= 0.10:
        return 'Small discount'
    elif d <= 0.25:
        return 'Medium discount'
    else:
        return 'Large discount'

assignment_df['discount_group'] = assignment_df['discount'].apply(classify_discount)

# 7. Potential data-quality flag
assignment_df['potential_data_issue'] = (
    assignment_df['revenue'].isna()
    | (assignment_df['quantity'] <= 0)
    | (assignment_df['shipping_cost'] < 0)
    | (assignment_df['product_name'].isna())
)

assignment_df.head()


## Assignment solution example

This is one possible solution. Students may create different features as long as they explain and justify them.


In [ ]:
solution_df = df.copy()

# Missing-value flags
solution_df['segment_missing'] = solution_df['segment'].isna()
solution_df['shipping_missing'] = solution_df['shipping_cost'].isna()
solution_df['revenue_missing'] = solution_df['revenue'].isna()

# Date features
solution_df['order_date'] = pd.to_datetime(solution_df['order_date'], errors='coerce')
solution_df['order_month'] = solution_df['order_date'].dt.month
solution_df['order_quarter'] = solution_df['order_date'].dt.quarter
solution_df['is_weekend'] = solution_df['order_date'].dt.dayofweek >= 5

# Numeric features
solution_df['gross_sales'] = solution_df['unit_price'] * solution_df['quantity']
solution_df['discount_amount'] = solution_df['gross_sales'] * solution_df['discount']
solution_df['estimated_net_sales'] = solution_df['gross_sales'] - solution_df['discount_amount']
solution_df['revenue_gap'] = solution_df['revenue'] - solution_df['estimated_net_sales']

# Text-cleaning features
solution_df['category_clean'] = solution_df['category'].astype('string').str.strip().str.lower()
solution_df['channel_clean'] = solution_df['channel'].astype('string').str.strip().str.lower().fillna('unknown')

# Group-based features
solution_df['category_avg_revenue'] = solution_df.groupby('category_clean')['revenue'].transform('mean')
solution_df['revenue_vs_category_avg'] = solution_df['revenue'] - solution_df['category_avg_revenue']

# Custom function feature
def classify_transaction(row):
    if pd.isna(row['revenue']):
        return 'Missing revenue'
    elif row['quantity'] <= 0:
        return 'Quantity issue'
    elif row['revenue'] >= 500 and row['discount'] > 0:
        return 'High revenue discounted'
    elif row['revenue'] >= 500:
        return 'High revenue'
    elif row['discount'] > 0:
        return 'Discounted'
    else:
        return 'Regular'

solution_df['transaction_type'] = solution_df.apply(classify_transaction, axis=1)

# Data-quality flag
solution_df['potential_data_issue'] = (
    solution_df['revenue'].isna()
    | (solution_df['quantity'] <= 0)
    | (solution_df['shipping_cost'] < 0)
    | (solution_df['product_name'].isna())
)

solution_df.head()


In [ ]:
# Summary of newly created columns
new_columns = [col for col in solution_df.columns if col not in df.columns]

feature_summary = pd.DataFrame({
    'new_column': new_columns,
    'missing_count': solution_df[new_columns].isna().sum().values,
    'dtype': solution_df[new_columns].dtypes.astype(str).values
})

feature_summary


In [ ]:
# Example business insights from engineered features
print('Example insights:')
print('- Rows with potential data issues:', solution_df['potential_data_issue'].sum())
print('- Average revenue by discount group:')
print(solution_df.groupby('transaction_type')['revenue'].mean().sort_values(ascending=False))
print('
- Revenue by weekend vs weekday:')
print(solution_df.groupby('is_weekend')['revenue'].sum())


# Final takeaway: cleaning plus feature engineering

Cleaning fixes the dataset. Feature engineering makes the dataset more useful.

A strong analyst does both:

1. detect and handle messy values;
2. preserve useful missingness signals;
3. create business meaningful columns;
4. check the new features carefully;
5. explain why each feature matters.


# Final takeaway

A good analyst does not treat all missing values the same. Some values can be filled, some should be flagged, some should be investigated, and some may justify dropping rows. The right decision depends on the business meaning of the column and the goal of the analysis.